# Masar Day 5 — Executed learner evidence

Extracted from the learner consolidated Colab notebook with the original executed code cells and retained outputs.


In [162]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_mtpvm923


In [163]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}


In [164]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {"zone_key": "Z_DAMMAM", "trip_count": 25, "total_fare_sar": "670.40"},
  {"zone_key": "Z_JEDDAH", "trip_count": 25, "total_fare_sar": "625.20"},
  {"zone_key": "Z_RIYADH", "trip_count": 25, "total_fare_sar": "585.00"}
]
AI feature example: {'zone_key': 'Z_DAMMAM', 

In [165]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day05_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file(): bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle: assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day05_handoff.zip
